In [1]:
#imports
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

from sklearn.metrics import classification_report, roc_auc_score, f1_score
import numpy as np
from tqdm import tqdm

import albumentations as A
from albumentations.pytorch import ToTensorV2

c:\CliniScan\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Cuda integration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
#dataset paths
DATA_DIR = r"C:\CliniScan\classification_data"

TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")

In [4]:
#Albumentations Augmentation Pipeline
train_transform = A.Compose([
    
    A.HorizontalFlip(p=0.5),

    A.Rotate(limit=10, p=0.5),

    A.RandomBrightnessContrast(
        brightness_limit=0.1,
        contrast_limit=0.1,
        p=0.5
    ),

    A.CLAHE(p=0.3),

    A.GaussNoise(var_limit=(5.0,20.0), p=0.2),

    A.Resize(224,224),

    A.Normalize(
        mean=(0.485,0.456,0.406),
        std=(0.229,0.224,0.225)
    ),

    ToTensorV2()
])

C:\Users\adith\AppData\Local\Temp\ipykernel_25892\1490643549.py:16: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5.0,20.0), p=0.2),


In [5]:

val_transform = A.Compose([
    
    A.Resize(224,224),

    A.Normalize(
        mean=(0.485,0.456,0.406),
        std=(0.229,0.224,0.225)
    ),

    ToTensorV2()
])

In [6]:
from PIL import Image

class AlbumentationsDataset(torch.utils.data.Dataset):

    def __init__(self, dataset, transform):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):

        img, label = self.dataset[idx]

        img = np.array(img)

        augmented = self.transform(image=img)

        img = augmented["image"]

        return img, label

In [7]:
base_train = ImageFolder(TRAIN_DIR)

base_val = ImageFolder(VAL_DIR)

train_dataset = AlbumentationsDataset(base_train, train_transform)

val_dataset = AlbumentationsDataset(base_val, val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

class_names = base_train.classes
num_classes = len(class_names)

print("Classes:", class_names)

Classes: ['Abnormal', 'Normal']


In [8]:
model = torchvision.models.resnet18(
    weights=torchvision.models.ResNet18_Weights.DEFAULT
)

model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)

In [9]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.0003)

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.1
)

EPOCHS = 10

best_val_acc = 0

In [10]:
torch.save(model.state_dict(), "classifier_augmented.pth")

In [11]:
all_labels = []
all_preds = []

model.eval()

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

In [12]:
print(classification_report(
    all_labels,
    all_preds,
    target_names=class_names
))

              precision    recall  f1-score   support

    Abnormal       0.30      0.71      0.42       836
      Normal       0.71      0.31      0.43      1990

    accuracy                           0.43      2826
   macro avg       0.51      0.51      0.43      2826
weighted avg       0.59      0.43      0.43      2826

